In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
from collections import defaultdict, deque, OrderedDict, Counter
from sklearn.preprocessing import LabelEncoder
from matplotlib import pyplot as plt
from scipy.stats import wasserstein_distance
from scipy.spatial import distance
import statistics

import copy
from tqdm.auto import tqdm
import pm4py


#Create 1-gram vector for Cosine and Euclidean distance

def createVector(charList):
    arrayList = np.array(charList)
    unique, counts = np.unique(arrayList, return_counts=True)
    
    #calculate relative frequency
    relFrequList = np.array((unique, counts)).T
    uniqueList = list(unique)
    return relFrequList[relFrequList[:, 0].argsort()]


        
#test


def alignArrays(array1, array2):
    
    commonSet = set(array1[:,0]).union(array2[:,0])
    
    val1 = array1[:,0] # NEW
    val2 = array2[:,0] # NEW
    
    for i in commonSet:
        if i not in val1:
            array1 = np.append(array1, np.array([[i, '0']]), axis=0)

    for i in commonSet:
        if i not in val2:
            array2 = np.append(array2, np.array([[i, '0']]), axis=0)
            
    return array1[array1[:, 0].argsort()], array2[array2[:, 0].argsort()]

#test
#t1 = ['a','b','c','d']
#t2 = ['c','d','e','f']
#vec1 = createVector(t1)
#vec2 = createVector(t2)
#alignArrays(vec1,vec2)

def activityList_multi(subLog):
    actList = []
    for i in range(len(subLog)):
        actList += subLog[i]
    return actList

# TEST
#subLog1 = logVar['concept:name'][0:10]
#activityList(subLog1)








def cosineSim1_multi(sub1,sub2):  

    #print(sub1)

    l1 = activityList_multi(sub1)
    l2 = activityList_multi(sub2)

    #print('entropy actList1:', calculateEntropy(l1))
    
    vec1 = createVector(l1)
    vec2 = createVector(l2)
    
    align1, align2 = alignArrays(vec1, vec2)
    
    a = align1[:,1].astype(int)
    b = align2[:,1].astype(int)
    
    return distance.cosine(a, b)
    
#test
#subLog1 = logVar['concept:name'][0:10].reset_index(drop=True)
#subLog2 = logVar['concept:name'][10:20].reset_index(drop=True)

#cosineSim1_multi(subLog1,subLog2)

#Create 2-gram

def df_list(list_of_char):
    extList = list_of_char.copy()
    extList.insert(0, '*') 
    extList.append('$')
    list_new = []
    for i in range(len(extList)):
        new = ' '.join(extList[i:i+2])
        list_new.append(new)
    del list_new[-1]
    return list_new
    
def df_list_multi(subLog):
    dfList = []
    for i in range(len(subLog)):
        dfList += df_list(subLog[i])
    return dfList

#Cosine distance based on 2-gram

def cosineSim2_multi(sub1,sub2):

    l1 = df_list_multi(sub1)
    l2 = df_list_multi(sub2)
    
    vec1 = createVector(l1)
    vec2 = createVector(l2)
    
    align1, align2 = alignArrays(vec1, vec2)
    
    a = align1[:,1].astype(int)
    b = align2[:,1].astype(int)
    
    return distance.cosine(a, b)



def calculateEntropy(featureList):
    counter = Counter(featureList)  # count the occurrences of each element in the window
    freqs = np.array(list(counter.values()))  # extract the frequency counts and store them as a numpy array
    freqs = freqs / freqs.sum()  # normalize the frequencies to obtain a probability distribution
    entropy = -np.sum(freqs * np.log(freqs)) / np.log(2) # compute the entropy of the probability distribution
    return entropy


def entropy1(sub):
    actList = activityList_multi(sub)
    return calculateEntropy(actList)

def entropy2(sub):
    graphList = graph_list_multi(sub)
    dfList = df_list_multi(sub)
    return calculateEntropy(dfList)


#test
#subLog1 = logVar['concept:name'][0:10].reset_index(drop=True)
#subLog2 = logVar['concept:name'][10:20].reset_index(drop=True)

#cosineSim2_multi(subLog1,subLog2)

# 2. transfer intList to int_tupleList

#Create tuple lists
def tuple_list(list_of_encodedActivities):
    #list.insert(0, '*')
    #list.append('*')
    list_new = []
    last_element = list_of_encodedActivities[-1]
    for i in range(len(list_of_encodedActivities)):
        new = tuple(list_of_encodedActivities[i:i+2])
        list_new.append(new)
    del list_new[-1]
    if list_of_encodedActivities.count(last_element) == 1: #check wether last activity in trace has some adjancency relation
        list_new.append((last_element,)) ### NOT Correct
    return list_new

#q = [0,0,0,0,1,1,2,3,4,5,3,2,4,0,5,6]
#tuple_list(q)
#logVar["int_tupleList"] = logVar["intList"].apply(lambda x: tuple_list(x))

# 3. generate Adjacency List

def adj_list(list_of_tuples):
    adj_list_new = {}
    try:
        for node1, node2 in list_of_tuples:
            #print(node1, node2)
            if node1 not in adj_list_new:
                newlist = []
                newlist.append(node2)
                adj_list_new[node1] = newlist
                #print(adj_list3)
        
            else:
                if node2 not in adj_list_new[node1]:
                    #mylist.extend(adj_list3[node1])
                    adj_list_new[node1].append(node2)
                    #print(adj_list3)
                    #adj_list3[node1] = mylist
    
    #in case activity has no adjacent activity - only possible for last activity --> tuple: (lastAct,)
    except ValueError as ve:
        lastValue = list_of_tuples[-1][0] 
        adj_list_new[lastValue] = list()
    return list(adj_list_new.values())

#q = [0,0,0,0,1,1,2,3,4,5,3,2,4,0,5,6]
#l = tuple_list(q)
#adj_list(l)

#logVar["int_adjList"] = logVar["int_tupleList"].apply(lambda x: adj_list(x))
#logVar["int_adjList"]


#Now consider length

def bfs_4(graph, start, end):
    
    graph = {v: k for v, k in enumerate(graph)}
    #print(start, end)
    queue = deque([(start, 0)])
    seen = set()
    while queue:
        #print(queue)
        node, distance = queue.popleft()
        #if not node:
            #print(start, end, queue)
            #print("GRAPH LIST", graph)
        if node in seen:
            continue
        seen.add(node)
        if node == end:
            return distance 
        for adjacent in graph.get(node, []):
            queue.append((adjacent, distance + 1))
        
#x = {0: [0, 1], 1: [2, 1, 0, int], 2:[2], [3: [1, 5, 3, 7], 4: [3], 5: [6, 5], 6: [1, 7], 7: [8, 9, 7], 8: [5, 8, 10], 9: [3]}
#y = [[0, 1, 5], [1, 2], [3, 4], [4, 2], [5, 0], [3, 6], []]
#bfs_4(y, 1, 6)

def reverse_graph(graph):
    reversed_graph = defaultdict(list)
    for node in graph:
        for neighbor in graph[node]:
            reversed_graph[neighbor].append(node)
    return reversed_graph


def bfs_5(graph, start, end):
    queue = deque([(start, 0)])
    seen = set()
    visited = {}
    while queue:
        node, distance = queue.popleft()
        if node in seen:
            continue
        seen.add(node)
        if node == end: # maybe quicker if adjacent directly checked
            return visited
        for adjacent in graph.get(node, []):
            queue.append((adjacent, distance + 1))
            if adjacent not in visited:
                visited.update({adjacent:distance})

            
def common_ancestors(graph, node1, node2): 
    #remove cross type edge between node1 and node2
    graph = copy.deepcopy(graph)
    graph[node1].remove(node2)
    graph = {v: k for v, k in enumerate(graph)}
    graphReverse = reverse_graph(graph)
    setNode1 = bfs_5(graphReverse, node1, 0)
    setNode2 = bfs_5(graphReverse, node2, 0)
    if next((a for a in list(setNode1) if a in list(setNode2)), None) == None:
        firstCommonAnces = next((a for a in list(setNode2) if a in list(setNode1)), None)
    else:
        firstCommonAnces = next((a for a in list(setNode1) if a in list(setNode2)), 0)
    
    #uses a hash map to identify the first common ancestor in both lists
    #looks for the first common ancestor in setNode1, which can also be found in setNode2 
    #--> this might not be the closest distance between setNode1 and setNode2
    #--> e.g., for x= [0,1,3,7,5,6] and y= [4,5,7,8,3] 7 might be closest ancestor, although algo detects 3 !
    #distance = setNode1[firstCommonAnces] + setNode2[firstCommonAnces]
    
    
    if firstCommonAnces != None:  
        ancesDistNode1 =  setNode1[firstCommonAnces] + 1 #the edge from node1 to first parent is counted as 0 by algorithm, therefore +1
        ancesDistNode2 =  setNode2[firstCommonAnces] + 1
        numberSkips = abs(ancesDistNode1 - ancesDistNode2)
        numberCross = min(ancesDistNode1, ancesDistNode2)
    else:
        numberSkips, numberCross = (0,1)
    return numberSkips, numberCross
    #if all(x in crossType for x in i):
    

    

#graphList = [[1], [2, 4, 1], [3, 2, 1], [], [5, 4], [5, 4, 6], [7], []]
#c = [[1, 4], [2], [3], [0, 5], [3, 5], []]
#c2 = {v: k for v, k in enumerate(c)}
#common_ancestors(c, 4, 5)
#reverse_graph(c2)

class Graph1:
    # instance variables
    def __init__(self, graph_list2, indexList):
        # v is the number of nodes/vertices
        self.time = 0
        self.traversal_array = []
        self.structural_array = [['sequ', 1]]
        #self.structural_array = []
        self.graph_list = graph_list2
        self.v = len(graph_list2)
        self.indexList = indexList

    # function for dfs
    def dfs(self):
        self.start_time = [-1]*self.v
        self.end_time = [-1]*self.v
 
        for node in range(self.v):
            if self.start_time[node] == -1:
                self.traverse_dfs(node)
                
        return np.array(self.structural_array)
        #print()
        #print("DFS Traversal: ", self.traversal_array)
        #print()
 
    def traverse_dfs(self, node):
        self.traversal_array.append(node)
        # get the starting time
        self.start_time[node] = self.time
        self.time += 1
        # traverse through the neighbours
        for neighbour in self.graph_list[node]:

            # when the neighbor was not yet visited
            if self.start_time[neighbour] == -1:                
                self.structural_array[0][1] += 0
                self.traverse_dfs(neighbour)
                
            # otherwise when the neighbour's visit is still ongoing:
            elif self.end_time[neighbour] == -1:
                if node == neighbour:
                    self.structural_array.append(['1back ',1])
                    #self.structural_array.append(['back ',1])
                    #self.structural_array.append([str(1)+'b'])
                
                elif node in self.graph_list[neighbour]:
                    #self.structural_array.append(['2back ',2])
                    self.structural_array.append(['2back ',1])
                    #self.structural_array.append(['back ',2])
                    #self.structural_array.append(str(2)+'b')
                    
                else:
                    x = bfs_4(self.graph_list, neighbour, node)
                    #self.structural_array.append([str(x+1)+'back ',x+1])
                    self.structural_array.append([str(x+1)+'back ',1])
                    #self.structural_array.append(['back ',x+1])
                    #self.structural_array.append(str(x+1)+'b')
                
            # otherwise when the neighbour's visit started before the current node's visit:
            elif self.start_time[node] < self.start_time[neighbour]:
                graph_list_copy = copy.deepcopy(self.graph_list)
                graph_list_copy[node].remove(neighbour)
                y = bfs_4(graph_list_copy, node, neighbour)
                #self.structural_array.append([str(y-1)+'forward ',y-1])
                self.structural_array.append([str(y-1)+'forward ',1])

            else:
                numberSkips, numberCross = common_ancestors(self.graph_list, node, neighbour)
                self.structural_array.append([str(numberCross)+'cross ',numberCross])
                self.structural_array.append([str(numberCross)+'cross ',1])
                
  

        self.end_time[node] = self.time
        self.time += 1#t1 = ['a','b','c','d']
#t2 = ['c','d','e','f']
#vec1 = createVector(t1)
#vec2 = createVector(t2)
#print('v1:',vec1, 'v2:',vec2)

def transform_list_of_pairs(pairs):
    return [pair[0] for pair in pairs]

def count_entries(input_list):
    # Count the occurrences of each unique entry in the list
    counter = Counter(input_list)
    
    # Create a NumPy array from the counter dictionary
    #result = np.array([[key, count] for key, count in counter.items()], dtype=object)
    
    return counter

def intEncoder(character_List):
    return [np.where(np.array(list(dict.fromkeys(character_List)))==e)[0][0]for e in character_List]

#Test
#input_list = ['sequ', '2back', '2back']
#result = count_entries(input_list)
#print(result)

def graph_list_multi(subLog):
    graphList = Counter([])
    for i in range(len(subLog)):
        v1 = subLog[i]
        #print('sublog number', i, v1)
        enc1 = intEncoder(v1)
        tuple1 = tuple_list(enc1)
        adj1 = adj_list(tuple1)
        index1 = list(OrderedDict.fromkeys(v1))
        graphF1 = Graph1(adj1,index1).dfs()
        list1 = transform_list_of_pairs(graphF1)
        # Count the occurrences of each unique entry in the list
        count1 = Counter(list1)
        graphList += count1
        #print(graphList)

    # Create a NumPy array from the counter dictionary
    result = np.array([[key, count] for key, count in graphList.items()], dtype=object)
    return result

#Alternative Option Concanate subtraces and apply graph analytics afterwards
def graph_list_multi2(subLog): 
    #graphList = Counter([])
    aggSubTrace = []
    for i in range(len(subLog)):
        aggSubTrace.extend(subLog[i])
       
    enc1 = intEncoder(aggSubTrace)
    tuple1 = tuple_list(enc1)
    adj1 = adj_list(tuple1)
    index1 = list(OrderedDict.fromkeys(aggSubTrace))
    graphF1 = Graph1(adj1,index1).dfs()
    list1 = transform_list_of_pairs(graphF1)
    # Count the occurrences of each unique entry in the list
    count1 = Counter(list1)
    #graphList += count1
    #print(graphList)

    # Create a NumPy array from the counter dictionary
    result = np.array([[key, count] for key, count in count1.items()], dtype=object)
    return result






#test
#x_test = pd.Series(logVar['concept:name'][0])
#t1 = x_test[0:5]
#t2 = x_test[5:10]
#graphSim1(t1,t2) 

def cosineGraphSim_multi(sub1,sub2):
    frequV1 = graph_list_multi(sub1)
    frequV2 = graph_list_multi(sub2)
    
    Vector1, Vector2 = alignArrays(frequV1, frequV2)

    a = Vector1[:,1].astype(int)
    b = Vector2[:,1].astype(int)
    
    return distance.cosine(a, b)

#test
#subLog1 = logVar['concept:name'][0:10].reset_index(drop=True)
#subLog2 = logVar['concept:name'][10:20].reset_index(drop=True)
#cosineGraphSim_multi(subLog1,subLog2)




# Calculate Entropy



#def entropy3(sub):
#    enc1 = intEncoder(sub)
#    tuple1 = tuple_list(enc1)
#    adj1 = adj_list(tuple1)
#    index1 = list(OrderedDict.fromkeys(v1))
#    graphF1 = Graph1(adj1,index1).dfs()
#    list1 = transform_list_of_pairs(graphF1)
#    return alculateEntropy(list1)#

C:\Users\la1949\AppData\Local\anaconda3\envs\ProcessMining\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
